In [1]:
%matplotlib inline
# Tell Jupyter to show plots inline (inside the notebook)

import xarray as xr          # N-dimensional labeled arrays for gridded data
import numpy as np           # Numerical arrays and math
import matplotlib.pyplot as plt  # Plotting
import matplotlib.patches as mpatches  # Used to build custom map legends
import cartopy.crs as ccrs   # Map coordinate reference systems
import cartopy.feature as cfeature  # Built-in geographic features for maps
import geopandas as gpd      # Spatial dataframes for shapefiles and geometries
import requests               # HTTP requests for downloading files
from shapely.geometry import Point  # Geometric point objects

import os                    # File system operations (e.g., checking file size)
import warnings
warnings.filterwarnings('ignore')  # Suppress minor deprecation warnings

print('All libraries loaded successfully.')
print(f'xarray version: {xr.__version__}')
print(f'geopandas version: {gpd.__version__}')

All libraries loaded successfully.
xarray version: 2026.4.0
geopandas version: 1.1.3


In [4]:
# OPeNDAP URL for gridMET maximum temperature, 2023
# xarray can open this URL directly -- it reads the file remotely over the network
opendap_url = (
    "http://thredds.northwestknowledge.net:8080/thredds/dodsC/"
    "MET/tmmx/tmmx_2023.nc"
)

In [5]:
#os needs to make the directory
os.makedirs("../data", exist_ok = True)
output_path = "../data/tmmx_2023_louisiana.nc"


In [6]:
#open remote dataset via OPeNDAP using xarray, which opens geospatial data
ds_remote = xr.open_dataset(opendap_url, engine = "netcdf4")
#this opens louisiana
ds_louisiana = ds_remote.sel(
    lat=slice(33.5, 28.5),   # Latitude: 33.5 N down to 28.5 N
    lon=slice(-94.5, -88.5)  # Longitude: 94.5 W to 88.5 W (negative = west)
)

ds_louisiana = ds_louisiana.load() #loads it into memory
ds_remote.close() #closes connection
if os.path.exists(output_path):
    os.remove(output_path)


# Write the in-memory dataset to a local NetCDF file
print(f"Saving to {output_path}...")
ds_louisiana.to_netcdf(output_path)

# Report the file size
size_mb = os.path.getsize(output_path) / 1e6
print(f"Done! File saved. Size on disk: {size_mb:.1f} MB")

# Open the local file -- all subsequent sections will use this variable 'ds'
ds = xr.open_dataset(output_path)
print("Local file loaded successfully. Here is a quick summary:")
print(ds)


Saving to ../data/tmmx_2023_louisiana.nc...
Done! File saved. Size on disk: 12.6 MB
Local file loaded successfully. Here is a quick summary:
<xarray.Dataset> Size: 50MB
Dimensions:          (day: 365, lat: 120, lon: 144, crs: 1)
Coordinates:
  * day              (day) datetime64[ns] 3kB 2023-01-01 ... 2023-12-31
  * lat              (lat) float64 960B 33.48 33.44 33.4 ... 28.61 28.57 28.52
  * lon              (lon) float64 1kB -94.47 -94.43 -94.39 ... -88.56 -88.52
  * crs              (crs) float32 4B 3.0
Data variables:
    air_temperature  (day, lat, lon) float64 50MB ...
Attributes: (12/19)
    geospatial_bounds_crs:      EPSG:4326
    Conventions:                CF-1.6
    geospatial_bounds:          POLYGON((-124.7666666333333 49.40000000000000...
    geospatial_lat_min:         25.066666666666666
    geospatial_lat_max:         49.40000000000000
    geospatial_lon_min:         -124.7666666333333
    ...                         ...
    date:                       02 March 2024
 

# NetCDF File

In [8]:

print("=== Full dataset summary ===")
print(ds)

=== Full dataset summary ===
<xarray.Dataset> Size: 50MB
Dimensions:          (day: 365, lat: 120, lon: 144, crs: 1)
Coordinates:
  * day              (day) datetime64[ns] 3kB 2023-01-01 ... 2023-12-31
  * lat              (lat) float64 960B 33.48 33.44 33.4 ... 28.61 28.57 28.52
  * lon              (lon) float64 1kB -94.47 -94.43 -94.39 ... -88.56 -88.52
  * crs              (crs) float32 4B 3.0
Data variables:
    air_temperature  (day, lat, lon) float64 50MB ...
Attributes: (12/19)
    geospatial_bounds_crs:      EPSG:4326
    Conventions:                CF-1.6
    geospatial_bounds:          POLYGON((-124.7666666333333 49.40000000000000...
    geospatial_lat_min:         25.066666666666666
    geospatial_lat_max:         49.40000000000000
    geospatial_lon_min:         -124.7666666333333
    ...                         ...
    date:                       02 March 2024
    note1:                      The projection information for this file is: ...
    note2:                      

In [9]:
print("=== Data variables ===")
for var_name in ds.data_vars:
    # Access the DataArray for this variable
    var = ds[var_name]
    print(f"\nVariable: {var_name}")
    print(f"  Dimensions: {var.dims}")
    print(f"  Shape: {var.shape}")
    print(f"  Data type: {var.dtype}")
    # Print all attributes (units, description, etc.)
    print("  Attributes:")
    for attr_key, attr_val in var.attrs.items():
        print(f"    {attr_key}: {attr_val}")

=== Data variables ===

Variable: air_temperature
  Dimensions: ('day', 'lat', 'lon')
  Shape: (365, 120, 144)
  Data type: float64
  Attributes:
    units: K
    description: Daily Maximum Temperature (2m)
    long_name: tmmx
    standard_name: tmmx
    dimensions: lon lat time
    grid_mapping: crs
    coordinate_system: WGS84,EPSG:4326
    _ChunkSizes: [ 61  98 231]


In [10]:
# --- Dimensions and their sizes ---

print("=== Dimensions and their sizes ===")
for dim_name, dim_size in ds.sizes.items():
    print(f"  {dim_name}: {dim_size} values")

# How many grid cells are in this dataset?
n_lat = ds.sizes['lat']   # Number of latitude steps
n_lon = ds.sizes['lon']   # Number of longitude steps
n_day = ds.sizes['day']   # Number of time steps
total_cells = n_lat * n_lon  # Spatial grid cells
total_values = n_lat * n_lon * n_day  # Total data values
print(f"\nSpatial grid cells (lat x lon): {n_lat} x {n_lon} = {total_cells:,}")
print(f"Total data values (lat x lon x day): {total_values:,}")

=== Dimensions and their sizes ===
  day: 365 values
  lat: 120 values
  lon: 144 values
  crs: 1 values

Spatial grid cells (lat x lon): 120 x 144 = 17,280
Total data values (lat x lon x day): 6,307,200


In [11]:
# --- Access the temperature variable and inspect its shape ---

# The actual temperature variable is named 'air_temperature' in the NetCDF file
# 'tmmx' is gridMET's short name for it (visible in the long_name attribute)
tmmx = ds['air_temperature']

print("=== Temperature variable: air_temperature ===")
print(f"Shape: {tmmx.shape}")
print(f"This means: ({tmmx.shape[0]} days, {tmmx.shape[1]} lat cells, "
      f"{tmmx.shape[2]} lon cells)")
print(f"Units: {tmmx.attrs['units']}")
print(f"Description: {tmmx.attrs['description']}")

# Show a slice: temperature on the first day, as a 2D array
print(f"\nTemperature on January 1, 2023 (first 3 rows, first 5 cols, in Kelvin):")
print(tmmx.isel(day=0).values[:3, :5].round(2))
print("  (Kelvin = Celsius + 273.15, so ~300 K is about 27 C)")

=== Temperature variable: air_temperature ===
Shape: (365, 120, 144)
This means: (365 days, 120 lat cells, 144 lon cells)
Units: K
Description: Daily Maximum Temperature (2m)

Temperature on January 1, 2023 (first 3 rows, first 5 cols, in Kelvin):
[[296.  295.9 295.6 295.7 295.6]
 [296.  296.  296.  295.9 295.7]
 [295.7 295.9 296.  296.  295.8]]
  (Kelvin = Celsius + 273.15, so ~300 K is about 27 C)


In [18]:
#get the air temperature for baton rouge
target_lat = 30
target_long = -91
pt = ds.sel(
    lat = target_lat,
    lon = target_long,
    method = 'nearest'
)
print(pt['air_temperature']) #full time series for this location
print(pt.data_vars) # what vars does it contain?

<xarray.DataArray 'air_temperature' (day: 365)> Size: 3kB
[365 values with dtype=float64]
Coordinates:
  * day      (day) datetime64[ns] 3kB 2023-01-01 2023-01-02 ... 2023-12-31
    lat      float64 8B 29.98
    lon      float64 8B -91.02
Attributes:
    units:              K
    description:        Daily Maximum Temperature (2m)
    long_name:          tmmx
    standard_name:      tmmx
    dimensions:         lon lat time
    grid_mapping:       crs
    coordinate_system:  WGS84,EPSG:4326
    _ChunkSizes:        [ 61  98 231]
Data variables:
    air_temperature  (day) float64 3kB ...
